In [ ]:
import pandas as pd
import json 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Multiple Dataset

In [ ]:
representative_order = [
    #Popular Representation
    'pipe_serialized',
    'token_serialized',
    'space_serialized',
    # Data Representation
    'csv',
    'tsv',
    'html',
    'markdown',
    'latex',
    'dict',
    'json',
    'xml',
    # Structural Transformations 
    'shuffled_rows',
    'shuffled_cols',
    'transpose',
    #Schema Definition Types
    'mschema',
    'macschema',
    'ddl',
    #Centroid
    "centroid_popular",
    "centroid_data",
    "centroid_schema",
    "centroid_structural",
     "centroid_all",
]
category = {"Popular Representation": ['pipe_serialized',    'token_serialized', 'space_serialized',"centroid_popular"],
    "Data Representation" : [
            'csv',
            'tsv',
            'html',
            'markdown',
            'latex',
            'dict',
            'json',
            'xml',
            'centroid_data'],
    "Structural Transformations": ['shuffled_rows',
        'shuffled_cols',
        'transpose',
        'centroid_structural'],
    "Schema Definition Types" :['mschema',
    'macschema',
    'ddl',
    'centroid_schema'],
    "All" :['centroid_all'],
}
category_wo_centroid = {"Popular Representation": ['pipe_serialized',    'token_serialized', 'space_serialized'],
    "Data Representation" : [
            'csv',
            'tsv',
            'html',
            'markdown',
            'latex',
            'dict',
            'json',
            'xml',
            ],
    "Structural Transformations": ['shuffled_rows',
        'shuffled_cols',
        'transpose'],
    "Schema Definition Types" :['mschema',
    'macschema',
    'ddl'],
}

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_list = []
for dataset in ["WTQ","WIKISQL","NQ"]:
    for model in ["splade","mpnet","bge","reasonir"]:#"reasonir",,"bge",
        df_model = None
        for idx,representative in enumerate(representative_order): 
            fpath = f'./data/retrieval_all/{model}_results_rank/{dataset}/{representative}_gold_rank_per_question.csv'
            df = pd.read_csv(fpath)
            df = df[["model","dataset","question_id","rank","hit@1"]]
            df_renamed = df.rename(columns={'rank': f'{representative}', 'hit@1': f'{representative}_hit@1'})
            if idx==0:
                df_model = df_renamed
            else:
                df_model = df_model.merge(df_renamed,on=["model","dataset","question_id"],how="inner",validate="one_to_one", ) 
        df_list.append(df_model)
df = pd.concat(df_list)    

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import ttest_rel
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

def pairwise_ttest_heatmap(
    g: pd.DataFrame,
    representative_order: list,
    *,
    score_suffix: str = "_gold_score",
    alpha: float = 0.01,
    correction: str = "fdr_bh",   # "fdr_bh", "bonferroni", etc. or None
    reorder_by: str = "mean",      # currently supports "mean"
    use_adjusted_for_plot: bool = True,
    pvalue_vmax: float = 0.10,     # cap heatmap scale to emphasize significance region
    annotate: bool = False,
    min_pairs: int = 2,
    title_prefix: str = "",
    show: bool = True,
):

    # 1) Validate and build score table
    score_cols = [f"{rep}{score_suffix}" for rep in representative_order]
    missing = [c for c in score_cols if c not in g.columns]
    if missing:
        raise ValueError(f"Missing expected score columns in group: {missing}")

    scores = g[score_cols].copy()
    scores.columns = representative_order

    # 2) Reorder
    if reorder_by != "mean":
        raise ValueError(f"Unsupported reorder_by='{reorder_by}'. Use 'mean'.")
    mean_perf = scores.mean(axis=0, skipna=True).sort_values(ascending=True)
    ordered_reps = mean_perf.index.tolist()
    scores = scores[ordered_reps]

    n = len(ordered_reps)
    pvals = pd.DataFrame(np.nan, index=ordered_reps, columns=ordered_reps)
    tstats = pd.DataFrame(np.nan, index=ordered_reps, columns=ordered_reps)
    mean_diff = pd.DataFrame(np.nan, index=ordered_reps, columns=ordered_reps)

    # For one-triangle correction
    raw_p_list = []
    pair_idx = []

    # 3) Pairwise paired t-tests
    for i in range(n):
        for j in range(n):
            if i == j:
                mean_diff.iat[i, j] = 0.0
                continue

            a = scores.iloc[:, i]
            b = scores.iloc[:, j]
            valid = a.notna() & b.notna()
            av = a[valid]
            bv = b[valid]

            if len(av) < min_pairs:
                continue

            stat, p = ttest_rel(av, bv, nan_policy="omit")
            tstats.iat[i, j] = stat
            pvals.iat[i, j] = p
            mean_diff.iat[i, j] = (av - bv).mean()

            if i > j:  # unique test
                raw_p_list.append(p)
                pair_idx.append((i, j))

    pvals_adj = pvals.copy()
    sig_mask = pvals_adj < alpha

    p_for_sig = pvals_adj if (correction is not None) else pvals
    p_for_plot = pvals_adj if (use_adjusted_for_plot and correction is not None) else pvals

    # 5) Summary and win counts
    summary = pd.DataFrame({
        "mean_score": mean_perf,
        "rank": np.arange(1, len(mean_perf) + 1),
    })

    win_counts = pd.Series(0, index=ordered_reps, dtype=int)
    for i, r in enumerate(ordered_reps):
        for j, c in enumerate(ordered_reps):
            if i == j:
                continue
            p = p_for_sig.iat[i, j]
            md = mean_diff.iat[i, j]
            if pd.notna(p) and pd.notna(md) and (p < alpha) and (md < 0):
                win_counts[r] += 1

    # 6) Plots
    fig_pvals = None
    fig_effect = None

    if show:
        # lower triangle only
        mask = np.triu(np.ones_like(p_for_plot, dtype=bool), k=0)

        fig, axes = plt.subplots(figsize=(16, 6),ncols=2)
        ax1,ax2 = axes[0],axes[1]
        # Effect size/direction heatmap
        mask2 = np.triu(np.ones_like(mean_diff, dtype=bool), k=0)
        sns.heatmap(
            mean_diff,
            mask=mask2,
            cmap="coolwarm",
            center=0,
            square=True,
            linewidths=0.5,
            vmin=-30,
            vmax=30,
            cbar_kws={"label": "Mean difference (row - col)"},
            annot=annotate,
            fmt=".3f",
            ax=ax1,
        )
        ax1.set_title(f"{title_prefix} Pairwise Mean Differences\n(lower triangle, reordered by mean performance)".strip())
        ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha="right")
        ax1.set_yticklabels(ax1.get_yticklabels(), rotation=0)
        sns.heatmap(
            p_for_plot,
            mask=mask,
            cmap="viridis_r",
            vmin=0,
            vmax=0.1,
            square=True,
            linewidths=0.5,
            cbar_kws={"label": "Adjusted p-value" if (use_adjusted_for_plot and correction is not None) else "Raw p-value"},
            annot=annotate,
            fmt=".3f",
            ax=ax2,
        )
        suffix = f" ({correction})" if (use_adjusted_for_plot and correction is not None) else " (raw)"
        ax2.set_title(f"{title_prefix} Pairwise Paired t-test p-values{suffix}\n(lower triangle, reordered by mean performance)".strip())
        ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha="right")
        ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0)


        fig.tight_layout()
        plt.show()

    return {
        "summary": summary,
        "pvals_raw": pvals,
        "pvals_adj": pvals_adj,
        "tstats": tstats,
        "mean_diff": mean_diff,
        "sig_mask": sig_mask,
        "win_counts": win_counts.sort_values(ascending=False),
        "ordered_reps": ordered_reps,
        "fig_pvals": fig_pvals,
        "fig_effect": fig_effect,
    }
def pairwise_wilcoxon_heatmap(
    g: pd.DataFrame,
    representative_order: list,
    *,
    score_suffix: str = "_gold_score",
    alpha: float = 0.01,
    correction: str = "fdr_bh",   # "fdr_bh", "bonferroni", etc. or None
    reorder_by: str = "mean",     # currently supports "mean"
    use_adjusted_for_plot: bool = True,
    pvalue_vmax: float = 0.10,    # cap heatmap scale to emphasize significance region
    annotate: bool = False,
    min_pairs: int = 2,
    title_prefix: str = "",
    show: bool = True,
    zero_method: str = "wilcox",  # "wilcox", "pratt", or "zsplit"
    alternative: str = "two-sided",
):
    # 1) Validate and build score table
    score_cols = [f"{rep}{score_suffix}" for rep in representative_order]
    missing = [c for c in score_cols if c not in g.columns]
    if missing:
        raise ValueError(f"Missing expected score columns in group: {missing}")

    scores = g[score_cols].copy()
    scores.columns = representative_order

    # 2) Reorder
    if reorder_by != "mean":
        raise ValueError(f"Unsupported reorder_by='{reorder_by}'. Use 'mean'.")
    mean_perf = scores.mean(axis=0, skipna=True).sort_values(ascending=True)
    ordered_reps = mean_perf.index.tolist()
    scores = scores[ordered_reps]

    n = len(ordered_reps)
    pvals = pd.DataFrame(np.nan, index=ordered_reps, columns=ordered_reps)
    wstats = pd.DataFrame(np.nan, index=ordered_reps, columns=ordered_reps)
    mean_diff = pd.DataFrame(np.nan, index=ordered_reps, columns=ordered_reps)

    # For one-triangle correction
    raw_p_list = []
    pair_idx = []

    # 3) Pairwise paired Wilcoxon signed-rank tests
    for i in range(n):
        for j in range(n):
            if i == j:
                mean_diff.iat[i, j] = 0.0
                continue

            a = scores.iloc[:, i]
            b = scores.iloc[:, j]
            valid = a.notna() & b.notna()
            av = a[valid]
            bv = b[valid]

            if len(av) < min_pairs:
                continue

            diffs = av - bv
            mean_diff.iat[i, j] = diffs.mean()

            # Wilcoxon can fail if all paired differences are zero
            if np.allclose(diffs.to_numpy(), 0):
                wstats.iat[i, j] = 0.0
                pvals.iat[i, j] = 1.0
                if i > j:
                    raw_p_list.append(1.0)
                    pair_idx.append((i, j))
                continue

            try:
                stat, p = wilcoxon(
                    av,
                    bv,
                    zero_method=zero_method,
                    alternative=alternative,
                    correction=False,
                    mode="auto",
                )
                wstats.iat[i, j] = stat
                pvals.iat[i, j] = p

                if i > j:  # unique test
                    raw_p_list.append(p)
                    pair_idx.append((i, j))

            except ValueError:
                # Fallback for edge cases such as insufficient non-zero differences
                continue

    # 4) Multiple testing correction on lower triangle, then mirror
    pvals_adj = pvals.copy()

    if correction is not None and len(raw_p_list) > 0:
        _, adj_pvals, _, _ = multipletests(raw_p_list, alpha=alpha, method=correction)

        for (i, j), p_adj in zip(pair_idx, adj_pvals):
            pvals_adj.iat[i, j] = p_adj
            pvals_adj.iat[j, i] = p_adj

    p_for_sig = pvals_adj if (correction is not None) else pvals
    p_for_plot = pvals_adj if (use_adjusted_for_plot and correction is not None) else pvals
    sig_mask = p_for_sig < alpha

    # 5) Summary and win counts
    summary = pd.DataFrame({
        "mean_score": mean_perf,
        "rank": np.arange(1, len(mean_perf) + 1),
    })

    win_counts = pd.Series(0, index=ordered_reps, dtype=int)
    for i, r in enumerate(ordered_reps):
        for j, c in enumerate(ordered_reps):
            if i == j:
                continue
            p = p_for_sig.iat[i, j]
            md = mean_diff.iat[i, j]
            if pd.notna(p) and pd.notna(md) and (p < alpha) and (md < 0):
                win_counts[r] += 1

    # 6) Plots
    fig_pvals = None
    fig_effect = None

    if show:
        mask = np.triu(np.ones_like(p_for_plot, dtype=bool), k=0)
        mask2 = np.triu(np.ones_like(mean_diff, dtype=bool), k=0)

        fig, axes = plt.subplots(figsize=(16, 6), ncols=2)
        ax1, ax2 = axes[0], axes[1]

        sns.heatmap(
            mean_diff,
            mask=mask2,
            cmap="coolwarm",
            center=0,
            square=True,
            linewidths=0.5,
            vmin=-30,
            vmax=30,
            cbar_kws={"label": "Mean Paired Rank Difference (row − col)"},
            annot=annotate,
            fmt=".3f",
            ax=ax1,
        )
        ax1.set_title(
            f"Average paired rank difference".strip()
        )
        ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha="right")
        ax1.set_yticklabels(ax1.get_yticklabels(), rotation=0)

        sns.heatmap(
            p_for_plot,
            mask=mask,
            cmap="viridis_r",
            vmin=0,
            vmax=pvalue_vmax,
            square=True,
            linewidths=0.5,
            cbar_kws={
                "label": "Adjusted p-value"
                if (use_adjusted_for_plot and correction is not None)
                else "Raw p-value"
            },
            annot=annotate,
            fmt=".3f",
            ax=ax2,
        )
        ax2.set_title(
            f"FDR-adjusted Wilcoxon p-values".strip()
        )
        ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha="right")
        ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0)

        fig.tight_layout()
        plt.show()

    return {
        "summary": summary,
        "pvals_raw": pvals,
        "pvals_adj": pvals_adj,
        "wstats": wstats,
        "mean_diff": mean_diff,
        "sig_mask": sig_mask,
        "win_counts": win_counts.sort_values(ascending=False),
        "ordered_reps": ordered_reps,
        "fig_pvals": fig_pvals,
        "fig_effect": fig_effect,
    }

In [ ]:
g = df
title = f"model=ALL"
out = pairwise_wilcoxon_heatmap(
    g,
    representative_order,
    score_suffix="",
    correction="holm",
    alpha=0.05,
    use_adjusted_for_plot=True,
    pvalue_vmax=0.10,
    title_prefix=title,
    show=True,    # set False if you only want returned matrices
)

# Example: print top performers in this group
print("\n" + "="*80)
print(title)
print(out["summary"].head(5))
print("Significant wins:")
print(out["win_counts"].head(10))

In [ ]:
results_by_group = {}
d="ALL"
for (m), g in df.groupby(["model"], dropna=False):
    title = f"model={m},dataset={d}"
    out = pairwise_ttest_heatmap(
        g,
        representative_order,
        score_suffix="",# "" means use rank, "_hit@1" means use hit@1
        correction=None,
        alpha=0.05,
        use_adjusted_for_plot=False,
        pvalue_vmax=0.10,
        title_prefix=title,
        show=True,    # set False if you only want returned matrices
    )
    results_by_group[(m, d)] = out

    # Example: print top performers in this group
    print("\n" + "="*80)
    print(title)
    print(out["summary"].head(5))
    print("Significant wins:")
    print(out["win_counts"].head(10))



In [ ]:
g = df
title = f"model=ALL"
out = pairwise_ttest_heatmap(
    g,
    representative_order,
    score_suffix="",
    correction=None,
    alpha=0.05,
    use_adjusted_for_plot=True,
    pvalue_vmax=0.10,
    title_prefix=title,
    show=True,    # set False if you only want returned matrices
)

# Example: print top performers in this group
print("\n" + "="*80)
print(title)
print(out["summary"].head(5))
print("Significant wins:")
print(out["win_counts"].head(10))
